# ALS — Alternating Least Squares (from scratch)

Sistema de recomendação de produtos para um supermercado online usando o **Online Retail Dataset** da UCI.

## Conteúdo
1. Imports
2. Carregamento dos dados
3. Exploração e limpeza
4. Construção da matriz cliente-produto
5. Implementação do ALS
6. Avaliação
7. Visualização das recomendações

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

## 2. Carregamento dos dados

O **Online Retail Dataset** da UCI contém transações reais de um supermercado online do Reino Unido entre 2010 e 2011. Possui ~500k transações de ~4k clientes e ~4k produtos.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

response = requests.get(url)
with open("online_retail.xlsx", "wb") as f:
    f.write(response.content)

df = pd.read_excel("online_retail.xlsx")
print(df.shape)
df.head()

## 3. Exploração e limpeza

Antes de construir a matriz cliente-produto, precisamos limpar os dados. Os problemas típicos neste dataset são:
- Clientes sem ID (compras anônimas — não podemos usá-las no ALS)
- Quantidades negativas (devoluções — não representam preferência)
- Produtos com descrição nula

In [ ]:
print("Shape original:", df.shape)
print("\nValores nulos por coluna:")
print(df.isnull().sum())

# Remover transações sem CustomerID — não podemos associá-las a nenhum cliente
df = df.dropna(subset=["CustomerID"])

# Remover devoluções (Quantity negativa) — não representam preferência real
df = df[df["Quantity"] > 0]

# Remover produtos sem descrição
df = df.dropna(subset=["Description"])

# Converter CustomerID para inteiro
df["CustomerID"] = df["CustomerID"].astype(int)

print("\nShape após limpeza:", df.shape)
df.head()

## 4. Construção da Matriz Cliente-Produto

Aqui transformamos as transações brutas na matriz **R** que o ALS vai fatorizar.

Cada célula `R[i, j]` contém o **total de unidades** que o cliente `i` comprou do produto `j`. Zeros representam produtos não comprados.

Esta matriz será muito **esparsa** — a maioria dos clientes compra apenas uma pequena fração de todos os produtos disponíveis.

In [ ]:
# Agregar: somar unidades por par (cliente, produto)
interactions = (
    df.groupby(["CustomerID", "StockCode"])["Quantity"]
    .sum()
    .reset_index()
)
interactions.columns = ["customer_id", "product_id", "quantity"]

# Construir a matriz R (clientes × produtos)
R = interactions.pivot_table(
    index="customer_id",
    columns="product_id",
    values="quantity",
    fill_value=0
)

R_matrix = R.values.astype(float)   # converter para numpy array

n_users, n_items = R_matrix.shape
print(f"Clientes: {n_users}")
print(f"Produtos: {n_items}")
print(f"Esparsidade: {(R_matrix == 0).sum() / R_matrix.size * 100:.1f}% de zeros")

# Visualizar uma amostra da matriz
sns.heatmap(R_matrix[:20, :30], cmap="YlOrRd")
plt.title("Amostra da matriz R (20 clientes × 30 produtos)")
plt.xlabel("Produto")
plt.ylabel("Cliente")
plt.show()

## 5. Implementação do ALS

### Como funciona

O ALS fatoriza **R** em duas matrizes:
- **U** — matriz de clientes `(n_clientes × k)` 
- **V** — matriz de produtos `(n_produtos × k)`

onde `k` é o número de **fatores latentes** (hiperparâmetro).

A cada iteração, alternamos:

**Passo 1 — Fixar V, resolver U:**
$$U_i = (V^T V + \lambda I)^{-1} V^T R_i$$

**Passo 2 — Fixar U, resolver V:**
$$V_j = (U^T U + \lambda I)^{-1} U^T R_j$$

O parâmetro `λ` (lambda) é a **regularização** — evita que o modelo memorize os dados de treino (overfitting).

In [ ]:
# Hiperparâmetros
K      = 20     # número de fatores latentes
LAMBDA = 0.1    # regularização — penaliza valores grandes em U e V
N_ITER = 20     # número de iterações (alternâncias U → V → U → ...)

def als(R: np.ndarray, k: int, lambda_: float, n_iter: int):
    """
    Alternating Least Squares.

    Args:
        R:        matriz cliente-produto  (n_users × n_items)
        k:        número de fatores latentes
        lambda_:  regularização
        n_iter:   número de iterações

    Returns:
        U: matriz de clientes  (n_users × k)
        V: matriz de produtos  (n_items × k)
    """
    n_users, n_items = R.shape

    # Inicializar U e V com valores aleatórios pequenos
    np.random.seed(42)
    U = np.random.normal(scale=0.1, size=(n_users, k))
    V = np.random.normal(scale=0.1, size=(n_items, k))

    I = np.eye(k)   # matriz identidade k×k (usada na regularização)
    loss_history = []

    for iteration in range(n_iter):

        # --- Passo 1: Fixar V, resolver U ---
        # Para cada cliente i: U[i] = (VᵀV + λI)⁻¹ Vᵀ R[i]
        VtV = V.T @ V                          # (k × k)
        for i in range(n_users):
            U[i] = np.linalg.solve(VtV + lambda_ * I, V.T @ R[i])

        # --- Passo 2: Fixar U, resolver V ---
        # Para cada produto j: V[j] = (UᵀU + λI)⁻¹ Uᵀ R[:,j]
        UtU = U.T @ U                          # (k × k)
        for j in range(n_items):
            V[j] = np.linalg.solve(UtU + lambda_ * I, U.T @ R[:, j])

        # Calcular loss: erro quadrático médio sobre interações conhecidas
        R_pred = U @ V.T
        mask   = R > 0                         # só avaliar onde há interações reais
        loss   = np.mean((R[mask] - R_pred[mask]) ** 2)
        loss_history.append(loss)

        print(f"Iteração {iteration + 1:02d}/{n_iter}  |  MSE: {loss:.4f}")

    return U, V, loss_history


U, V, loss_history = als(R_matrix, k=K, lambda_=LAMBDA, n_iter=N_ITER)

## 6. Avaliação

Avaliamos o modelo com o **MSE** (Mean Squared Error) — o mesmo erro que minimizamos durante o treino.

Também visualizamos a curva de convergência: o MSE deve cair a cada iteração e estabilizar quando o modelo convergiu.

In [ ]:
plt.plot(range(1, N_ITER + 1), loss_history, marker="o")
plt.xlabel("Iteração")
plt.ylabel("MSE")
plt.title("Convergência do ALS")
plt.grid(True)
plt.show()

print(f"MSE final: {loss_history[-1]:.4f}")

## 7. Recomendações

Para gerar recomendações para um cliente:

1. Calcular os scores para todos os produtos: `scores = U[cliente] @ V.T`
2. Excluir produtos que o cliente já comprou
3. Recomendar os produtos com maior score

In [ ]:
def recomendar(user_idx: int, U: np.ndarray, V: np.ndarray,
               R: np.ndarray, product_names: list, n: int = 10):
    """
    Gera top-N recomendações para um cliente.

    O score de cada produto é o produto escalar entre o vetor
    do cliente U[i] e o vetor do produto V[j] — quanto maior,
    mais alinhado o produto está com as preferências do cliente.
    """
    # Calcular scores para todos os produtos
    scores = U[user_idx] @ V.T                    # (n_items,)

    # Mascarar produtos já comprados — não queremos recomendá-los
    already_bought = R[user_idx] > 0
    scores[already_bought] = -np.inf

    # Pegar os índices dos top-N scores
    top_indices = np.argsort(scores)[::-1][:n]

    print(f"Top {n} recomendações para o cliente {user_idx}:\n")
    for rank, idx in enumerate(top_indices, 1):
        print(f"  {rank:2}. {product_names[idx]}  (score: {scores[idx]:.2f})")


product_names = R.columns.tolist()
recomendar(user_idx=0, U=U, V=V, R=R_matrix, product_names=product_names)